# 01 · LLMOps en práctica · HuggingFace -> Cuantización 4-bit

> Demo práctica del módulo de LLMOps. Cargas un LLM moderno, lo cuantizas a 4-bit para servirlo más barato.
> Dos pasos, un mismo modelo. Todo en Colab.

## Qué aprendemos

1. **HuggingFace + Gradio** · cargar un Qwen3-4B-Instruct y montar una demo de chat en 4 líneas.
2. **Chat template moderno** · `tokenizer.apply_chat_template()` (sin marcadores manuales).
3. **Cuantización 4-bit (NF4)** · misma calidad, ~4x menos VRAM. La técnica clave de LLMOps.
4. **Matriz local vs API** · cuándo bajar a local y cuándo seguir pagando tokens.

## Por qué importa

Servir tu propio modelo es lo que rompe la dependencia (y la factura) de las APIs. Pasar de HF -> 4-bit es el primer paso de LLMOps para servir un open-source más barato.

## Conecta con

- **Fine-tuning** · una vez sabes servir, el siguiente paso es entrenar el modelo en tu dominio.
- **Lab 05 de Fine-tuning (distillation)** · el argumento de coste local vs API.
- **Slides de LLMOps** · el lifecycle: estás en LLMOps / producción.

## Stack

`transformers` · `bitsandbytes` (4-bit) · `gradio`.

## Hardware

T4 (Colab Free). El 4-bit cabe de sobra.


---

## Setup

Instalamos `transformers`, `accelerate`, `bitsandbytes` (para la parte de cuantización) y `gradio` (para la demo web).


In [1]:
!pip install -qU transformers accelerate bitsandbytes gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/43.1 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 35.4/43.1 MB 126.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 125.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 125.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 19.1/31.4 MB 121.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.6 MB/s eta 0:00:00


### Conexión con HuggingFace

Muchos modelos open-source requieren aceptar su licencia antes de descargarlos. Necesitas:

1. Cuenta en HuggingFace.
2. Aceptar la licencia del modelo Qwen3 (no es gated por defecto, pero hay variantes que sí).
3. Token de acceso desde [settings/tokens](https://huggingface.co/settings/tokens).


In [2]:
# Login a HuggingFace (token desde huggingface.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


---

# Parte A · HuggingFace + Gradio · la ruta más rápida a una demo

Lo más rápido para mostrarle un LLM a tu cliente:

- **HuggingFace** · el registry · catálogo de modelos + pesos + tokenizers.
- **Gradio** · librería de Python que envuelve una función en una web UI lista para compartir.

3 pasos · 4 líneas reales de código en cada paso.


## A.1 · Cargar el modelo

Qwen3-4B-Instruct cabe en una T4 gratis de Colab en `bfloat16` (~8 GB VRAM).


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

print(f"Parámetros: {model.num_parameters() / 1e9:.2f}B")
print(f"VRAM usada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Parámetros: 4.02B
VRAM usada: 7.49 GB


## A.2 · Chat template moderno

Qwen3 (como Llama 3, Mistral, Gemma 4, etc.) trae su `chat_template` definido en el tokenizer. **No** se construye el prompt a mano con marcadores tipo `[INST]` (eso era de Llama 2).

La receta universal hoy:

```python
messages = [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

`apply_chat_template` lee el `chat_template` del `tokenizer_config.json` y aplica el formato correcto. Si cambias Qwen3 por Llama 3 o Gemma 4, el código sigue igual · es la abstracción que mantiene tu pipeline portable.


In [4]:
messages = [
    {"role": "system", "content": "Eres un asistente conciso. Respondes en menos de 50 palabras."},
    {"role": "user", "content": "¿Qué es LLMOps en una frase?"},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(response)

LLMOps es la gestión de modelos de lenguaje grande (LLMs) desde su desarrollo hasta producción, asegurando escalabilidad, calidad y mantenimiento en entornos reales.


## A.3 · Envolverlo con Gradio · demo en 4 líneas

`gr.ChatInterface(fn).launch(share=True)` toma una función `(message, history) -> response` y construye una web de chat. `share=True` te da una URL pública temporal para enseñar la demo desde el portátil.


In [5]:
SYSTEM_PROMPT = "Eres un asistente útil. Respondes con claridad y concisión."


def chat(message: str, history: list) -> str:
    """Función chat para gr.ChatInterface.

    En Gradio 5/6 `history` llega en formato mensajes: [{"role": "user", "content": ...}, ...]
    (el antiguo formato de tuplas (user, bot) ya no existe). Nos quedamos con role + texto.
    """
    def _texto(c):   # content puede ser str o una lista de bloques {"type": "text", "text": ...}
        if isinstance(c, str):
            return c
        return " ".join(b.get("text", "") for b in c if isinstance(b, dict))

    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for m in history:
        msgs.append({"role": m["role"], "content": _texto(m["content"])})
    msgs.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


# Prueba rápida sin UI: un segundo turno con historial (así sabes que la función va antes de lanzar Gradio)
print(chat("¿Y en dos palabras?", [
    {"role": "user", "content": "¿Qué es LLMOps en una frase?"},
    {"role": "assistant", "content": "Llevar LLMs a producción y operarlos de forma fiable."},
]))

Operar LLMs


In [6]:
import gradio as gr

gr.ChatInterface(chat, title="Qwen3-4B · Chat").launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


* Running on public URL: https://ab482a293f3d04ab7d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

# Parte B · Cuantización 4-bit (NF4) · el caballo de batalla de LLMOps

Reducir la precisión de los pesos de bf16 (16 bits) a int4 (4 bits) **divide la VRAM por ~4** con pérdida de calidad despreciable. Es el truco que hace que este 4B pase de ~7,5 GB a ~2,5 GB (lo mides abajo) o que un 8B quepa en una GPU de consumo de 8 GB. Ojo con la escala: un 70B en 4-bit sigue pesando ~35-40 GB, así que NO cabe en una RTX 4090 (24 GB); necesitas dos 4090 o una GPU de 48-80 GB.

La receta canónica: **NF4 + double quantization + bf16 compute** (lo que usa el paper QLoRA, hoy estándar de facto).


## B.1 · Liberar memoria

Antes de cargar el segundo modelo, soltamos el primero · si no, dos modelos a la vez nos sacan de la T4.


In [7]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"VRAM tras liberar: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

VRAM tras liberar: 0.01 GB


## B.2 · Cargar el mismo modelo en 4-bit

`BitsAndBytesConfig` hace el trabajo. 4 flags clave:

| Flag | Qué hace |
|---|---|
| `load_in_4bit=True` | Activa la cuantización 4-bit |
| `bnb_4bit_quant_type="nf4"` | NF4 (NormalFloat 4) · distribución optimizada para pesos normales |
| `bnb_4bit_use_double_quant=True` | Doble cuantización (las constantes también se cuantizan) · ~7% menos VRAM |
| `bnb_4bit_compute_dtype=torch.bfloat16` | Las multiplicaciones se hacen en bf16 (precisión que vale la pena pagar en compute) |


In [8]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"VRAM con NF4 4-bit: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

VRAM con NF4 4-bit: 2.50 GB


## B.3 · Misma pregunta · misma respuesta

Si la cuantización es buena, el modelo 4-bit debería responder esencialmente lo mismo que el bf16 (ligeras diferencias por el sampling estocástico). Comparemos:


In [9]:
messages = [
    {"role": "system", "content": "Eres un asistente conciso. Respondes en menos de 50 palabras."},
    {"role": "user", "content": "¿Qué es LLMOps en una frase?"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model_4bit.device)

out = model_4bit.generate(**inputs, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

LLMOps es la gestión operativa de modelos de lenguaje grande, que incluye su desarrollo, deploy, monitoreo y escalabilidad en entornos productivos.


---

# Cierre · Matriz de decisión · local vs API

Ya tienes el mismo modelo en 2 modos (precisión completa y 4-bit). ¿Local o API en tu proyecto? Depende del caso de uso.


In [10]:
import pandas as pd

matriz = pd.DataFrame([
    {
        "Caso de uso": "PoC en 1 semana",
        "Mejor opción": "API (OpenAI / Anthropic)",
        "Por qué": "Sin setup, mejor calidad de salida desde el minuto 0",
    },
    {
        "Caso de uso": "Volumen alto sostenido (>100k req/mes)",
        "Mejor opción": "Local (GGUF / llama.cpp)",
        "Por qué": "Coste fijo en electricidad, no por token. Ver lab 05 de Fine-tuning (distillation moat)",
    },
    {
        "Caso de uso": "Datos sensibles (legal, salud, banca)",
        "Mejor opción": "Local",
        "Por qué": "Los datos no salen del dispositivo · compliance + EU AI Act",
    },
    {
        "Caso de uso": "Multimodal frontera (vision, voz, razonamiento)",
        "Mejor opción": "API",
        "Por qué": "GPT-5, Claude Opus 4.7 y Gemini 3 siguen liderando la frontera",
    },
    {
        "Caso de uso": "Demo interna / prototipo rápido",
        "Mejor opción": "HF + Gradio (Parte A)",
        "Por qué": "URL pública en 4 líneas · sin desplegar nada",
    },
    {
        "Caso de uso": "Edge (móvil, IoT, sin conexión)",
        "Mejor opción": "Local cuantizado",
        "Por qué": "Solo Ollama / llama.cpp / MLX pueden correr sin servidor",
    },
])

matriz

,Caso de uso,Mejor opción,Por qué
0,PoC en 1 semana,API (OpenAI / Anthropic),"Sin setup, mejor calidad de salida desde el mi..."
1,Volumen alto sostenido (>100k req/mes),Local (GGUF / llama.cpp),"Coste fijo en electricidad, no por token. Ver ..."
2,"Datos sensibles (legal, salud, banca)",Local,Los datos no salen del dispositivo · complianc...
3,"Multimodal frontera (vision, voz, razonamiento)",API,"GPT-5, Claude Opus 4.7 y Gemini 3 siguen lider..."
4,Demo interna / prototipo rápido,HF + Gradio (Parte A),URL pública en 4 líneas · sin desplegar nada
5,"Edge (móvil, IoT, sin conexión)",Local cuantizado,Solo Ollama / llama.cpp / MLX pueden correr si...


---

## Recap

Has visto el mismo Qwen3-4B en 2 modos:

1. **HF + transformers (Parte A)** · ~8 GB VRAM · python-first · ideal para desarrollo y fine-tuning.
2. **HF + bitsandbytes 4-bit (Parte B)** · ~2.5 GB VRAM · misma calidad · ideal para servir barato.

## Hacia Fine-tuning

Ahora sabes **servir** un modelo open-source en 2 modos. La pregunta siguiente: **¿cómo entrenas un modelo abierto (Qwen 3.5, Gemma 4) para que sepa tu dominio?**

Eso es el módulo de Fine-tuning. Los 3 anclas:

- **Lab 04 [ANCLA]** · QLoRA + Unsloth sobre Qwen3.5-2B · le das la voz de Milei con 447 conversaciones, en una T4 gratis.
- **Lab 05 [ANCLA]** · Distillation · `gpt-5-mini` (teacher por API) genera 200 Q&A de Python y Gemma 4 E2B (student, QLoRA en T4) lo aprende para responder en local.
- **Lab 07 [ANCLA]** · DPO + ORPO · alineas el modelo con preferencias humanas.

Cuando termines Fine-tuning, tendrás un adapter LoRA sobre un modelo base en 4-bit: lo cargas con lo que has visto aquí (transformers + bitsandbytes) y lo sirves en local. El círculo se cierra.
